In [0]:
%sql
-- - Maior Streak
-- - Share de dia da semana, mês e semana do ano
-- - Tempo entre as interações
-- - Quantidade de episódios finalizados nos últimos  x dias ou vida toda

In [0]:
%sql
-- Maior Streak

WITH
tb_transaction_period AS (
    SELECT
        *,
        DATE(DtCriacao) AS DtDia
    FROM workspace.tmw_loyalty.transacoes
    WHERE DtCriacao < '2026-06-01'
        AND DtCriacao >= '2026-06-01' - interval 28 days
) ,

tb_product_full AS (
    SELECT
        t1.*,
        t2.idTransacaoProduto,
        t2.QtdeProduto,
        t2.vlProduto,
        t3.DescCategoriaProduto,
        t3.DescDescricaoProduto,
        t3.DescNomeProduto
    FROM tb_transaction_period AS t1
    LEFT JOIN workspace.tmw_loyalty.transacao_produto AS t2
        ON t1.IdTransacao = t2.IdTransacao
    LEFT JOIN workspace.tmw_loyalty.produtos AS t3
        ON t2.IdProduto = t3.IdProduto
) ,

tb_max_streak AS (
    SELECT
        IdCliente,
        DescNomeProduto,
        MAX(QtdePontos) AS MaxStreak,
        MAX(QtdePontos)/100 AS MaxStreak_vezes       
    FROM tb_product_full
    WHERE LOWER(DescNomeProduto) LIKE "%streak%"
    GROUP BY ALL
)

SELECT
    *
FROM tb_max_streak
ORDER BY MaxStreak DESC

In [0]:
%sql
-- Share de dia da semana, mês e semana do ano
WITH
tb_transaction_period AS (
    SELECT
        *,
        DATE(DtCriacao) AS DtDia,
        DAYOFWEEK(DATE(DtCriacao)) AS DiaDaSemana, -- 1 = Domingo, 7 = Sábado
        DAYOFYEAR(DATE(DtCriacao)) AS DiaDoAno,
        MONTH(DATE(DtCriacao)) AS Mes,
        WEEKOFYEAR(DATE(DtCriacao)) AS SemanaDoAno,
        FLOOR(DATEDIFF('2026-06-01', DATE(DtCriacao)) / 7) + 1 AS SemanaIdx -- 1 = most recent week, 4 = oldest week
    FROM workspace.tmw_loyalty.transacoes
    WHERE DtCriacao < '2026-06-01'
        AND DtCriacao >= '2026-06-01' - interval 28 days
),

-- Denominador: total de transações por cliente
tb_total AS (
    SELECT
        IdCliente,
        COUNT(*) AS QtdeTotal
    FROM tb_transaction_period
    GROUP BY IdCliente
),

-- Numerador: contagens por cliente × dia da semana
tb_dia AS (
    SELECT
        IdCliente,
        COUNT(CASE WHEN DiaDaSemana = 1 THEN 1 END) AS QtdeDom,
        COUNT(CASE WHEN DiaDaSemana = 2 THEN 1 END) AS QtdeSeg,
        COUNT(CASE WHEN DiaDaSemana = 3 THEN 1 END) AS QtdeTer,
        COUNT(CASE WHEN DiaDaSemana = 4 THEN 1 END) AS QtdeQua,
        COUNT(CASE WHEN DiaDaSemana = 5 THEN 1 END) AS QtdeQui,
        COUNT(CASE WHEN DiaDaSemana = 6 THEN 1 END) AS QtdeSex,
        COUNT(CASE WHEN DiaDaSemana = 7 THEN 1 END) AS QtdeSab
    FROM tb_transaction_period
    GROUP BY IdCliente
),

-- Numerador: contagens por cliente × semana relativa
tb_semana AS (
    SELECT
        IdCliente,
        COUNT(CASE WHEN SemanaIdx = 1 THEN 1 END) AS QtdeSem1,
        COUNT(CASE WHEN SemanaIdx = 2 THEN 1 END) AS QtdeSem2,
        COUNT(CASE WHEN SemanaIdx = 3 THEN 1 END) AS QtdeSem3,
        COUNT(CASE WHEN SemanaIdx = 4 THEN 1 END) AS QtdeSem4
    FROM tb_transaction_period
    GROUP BY IdCliente
)

SELECT
    t.IdCliente,
    t.QtdeTotal,

    -- ---------- Shares por dia da semana (%) ----------
    ROUND(d.QtdeDom * 100.0 / t.QtdeTotal, 2) AS ShareDomingo,
    ROUND(d.QtdeSeg * 100.0 / t.QtdeTotal, 2) AS ShareSegunda,
    ROUND(d.QtdeTer * 100.0 / t.QtdeTotal, 2) AS ShareTerca,
    ROUND(d.QtdeQua * 100.0 / t.QtdeTotal, 2) AS ShareQuarta,
    ROUND(d.QtdeQui * 100.0 / t.QtdeTotal, 2) AS ShareQuinta,
    ROUND(d.QtdeSex * 100.0 / t.QtdeTotal, 2) AS ShareSexta,
    ROUND(d.QtdeSab * 100.0 / t.QtdeTotal, 2) AS ShareSabado,

    -- ---------- Shares por semana relativa (%) ----------
    ROUND(s.QtdeSem1 * 100.0 / t.QtdeTotal, 2) AS ShareSemana1,  -- mais recente
    ROUND(s.QtdeSem2 * 100.0 / t.QtdeTotal, 2) AS ShareSemana2,
    ROUND(s.QtdeSem3 * 100.0 / t.QtdeTotal, 2) AS ShareSemana3,
    ROUND(s.QtdeSem4 * 100.0 / t.QtdeTotal, 2) AS ShareSemana4  -- mais antiga

FROM tb_total AS t
LEFT JOIN tb_dia As d 
    ON d.IdCliente = t.IdCliente
LEFT JOIN tb_semana AS s 
    ON s.IdCliente = t.IdCliente
ORDER BY t.IdCliente;

In [0]:
%sql
-- - Tempo entre as interações
WITH
tb_transaction_period AS (
    SELECT
        *,
        DATE(DtCriacao) AS DtDia
    FROM workspace.tmw_loyalty.transacoes
    WHERE DtCriacao < '2026-06-01'
        AND DtCriacao >= '2026-06-01' - interval 28 days
),

tb_client_day_gap AS (
    SELECT
        IdCliente,
        DtDia,
        MAX(DtCriacao) AS MaxDt,
        MIN(DtCriacao) AS MinDt,
        (UNIX_TIMESTAMP(MAX(DtCriacao)) - UNIX_TIMESTAMP(MIN(DtCriacao))) / 3600.0 AS GapHoras,
        (UNIX_TIMESTAMP(MAX(DtCriacao)) - UNIX_TIMESTAMP(MIN(DtCriacao))) / 60.0   AS GapMinutos
    FROM tb_transaction_period
    GROUP BY ALL
)

SELECT
    IdCliente,
    COUNT(*) AS DaysCount,
    AVG(GapHoras) AS AvgGapHoras,
    AVG(GapMinutos) AS AvgGapMinutos
FROM tb_client_day_gap
GROUP BY IdCliente
ORDER BY DaysCount DESC